# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain.agents import create_agent

In [8]:
agent = create_agent(
    model="amazon.nova-pro-v1:0",
    system_prompt="You are a full-stack comedian",
)

## No Steaming (invoke)

In [9]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Alright, here's a joke that's full-stack:

Why did the server break up with the database?

Because it couldn't handle the commitment! The server said, "It's just too much pressure. Every time I try to query you, you're either too slow or throw an error. I need someone who can handle my requests without crashing. Plus, you're always so normalized—it's like you don't even want to join with me!"

Hope that one didn’t throw any errors in your humor stack!


## values
You have seen this streaming mode in our examples so far. 

In [10]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Alright, here's a dad joke for you:

Why did the scarecrow win an award?

Because he was outstanding in his field! 🌾🏆

Hope that brought a smile to your face!


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [ ]:
# for token, metadata in agent.stream(
#     {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
#     stream_mode="messages",
# ):
#     print(f"{token.content}", end="")

In [13]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    # Check if content is a list and has items (common with Bedrock)
    if isinstance(token.content, list) and len(token.content) > 0:
        # Extract text from the first block
        print(token.content[0].get("text", ""), end="")
    else:
        # Fallback for standard string content
        print(f"{token.content}", end="")

[]Sure, here's a family-friendly poem for you:

In a world of wonder, where dreams take flight,
Where laughter echoes and everything's bright,
There's a place we gather, a haven of cheer,
A family united, with love drawing near.

Mornings arise with the sun's gentle kiss,
As we share our moments, our joys and our bliss.
Breakfast tables chatter, a symphony of sound,
In this sacred space, where love is profound.

Adventures await in the great outdoors,
Nature's embrace, where happiness soars.
Picnics in meadows, laughter in the air,
Creating memories, beyond compare.

Evenings unfold with stories to tell,
Of days filled with magic, and moments that swell.
Cuddles and whispers, secrets shared,
In the warmth of togetherness, we find ourselves cared.

Through storms and through sunshine, we stand side by side,
A family resilient, with love as our guide.
In the tapestry of life, we weave our own thread,
A story of unity, in words never said.

So here's to the family, a bond ever true,
A sou

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [14]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="amazon.nova-lite-v1:0",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='74dace14-861c-4e8a-95f0-814d2960ca08')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='74dace14-861c-4e8a-95f0-814d2960ca08'), AIMessage(content=[{'type': 'text', 'text': "<thinking>The User wants to know the weather in San Francisco (SF). I need to use the 'get_weather' tool for this. I'll call this tool with the city parameter set to 'SF'.</thinking>\n"}, {'type': 'tool_use', 'name': 'get_weather', 'input': {'city': 'SF'}, 'id': 'tooluse_UOPOj9zDSNyj0t_-268m-w'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '9133c8d0-447c-4722-83fc-eab3f71e4975', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 18 Dec 2025 02:28:34 GMT', 'content-type': 'application/json', 'content-length': '490', 'connection': 'keep-alive', 'x-amzn-requestid': '9133c8d0-447c-4722-83

In [15]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

: 